In [8]:
import pandas as pd
import numpy as np
import os
import time
import datetime
from datetime import date
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from scipy.optimize import minimize
# from pykalman import KalmanFilter
# from Vasicekcalibration import calibrate_vasicek

from helpers.data_helpers import load_calibration_data
from helpers.plot_helpers import plot_term_structure, plot_term_structure_evolution
from helpers.nelson_curve_helpers import NSS_rate, NSS_estimation, plot_nss_fits

# NSS model

In [10]:
commo_ticker = 'KC'  # Coffee
steepener_data = load_calibration_data(commo_ticker=commo_ticker)
# steepener_data.count()

Loading commodity data took 1.70 seconds.
Calculating time to maturity took 1.05 seconds.
Loading short rate data took 0.02 seconds.


In [ ]:
def nss_metrics_for_date(
    df: pd.DataFrame,
    date_str,
    min_maturity: int = 20,
    max_maturity: int = 250,
    verbosity: bool = False,
):
    """
    Fit NSS for a given date and return:
      - params
      - MAE(log)
      - RMSE(log)
      - sMAPE (%)

    Assumptions:
      - df indexed by date
      - columns: 'time_to_maturity', 'price'
      - NSS_rate(maturities, params) returns LOG(prices)
      - NSS_estimation(maturities, prices, ...) fits accordingly
    """
    if date_str not in df.index:
        raise ValueError(f"{date_str} not found in DataFrame index.")

    day_data = df.loc[date_str]
    if isinstance(day_data, pd.Series):
        day_data = day_data.to_frame().T

    day_data = day_data[
        (day_data["time_to_maturity"] >= min_maturity) &
        (day_data["time_to_maturity"] <= max_maturity)
    ].sort_values("time_to_maturity")

    maturities = day_data["time_to_maturity"].to_numpy()
    prices = day_data["price"].to_numpy()

    if len(maturities) < 3:
        raise ValueError("Not enough contracts after filtering to compute NSS metrics.")

    # Fit NSS
    params = NSS_estimation(
        maturities,
        prices,
        min_maturity=min_maturity,
        max_maturity=max_maturity
    )

    # Preds in log-space and price-space
    log_prices = np.log(np.maximum(prices, 1e-12))
    log_modeled = NSS_rate(maturities, params)
    modeled_prices = np.exp(log_modeled)

    # ---- MAE(log), RMSE(log) ----
    e_log = log_prices - log_modeled
    mae_log = float(np.mean(np.abs(e_log)))
    rmse_log = float(np.sqrt(np.mean(e_log ** 2)))

    # ---- sMAPE (%) ----
    denom = np.maximum(np.abs(prices) + np.abs(modeled_prices), 1e-12)
    smape = float(100.0 * np.mean(2.0 * np.abs(prices - modeled_prices) / denom))

    if verbosity:
        d = pd.Timestamp(date_str).date() if not isinstance(date_str, pd.Timestamp) else date_str.date()
        print(f"\nDate: {d}")
        print(f"Contracts used: {len(maturities)}")
        print("Maturities:", maturities)
        print("Actual prices:", np.round(prices, 6))
        print("Modeled prices:", np.round(modeled_prices, 6))
        print(f"MAE(log):  {mae_log:.6f}")
        print(f"RMSE(log): {rmse_log:.6f}")
        print(f"sMAPE(%):  {smape:.4f}")

    metrics = {
        "mae_log": mae_log,
        "rmse_log": rmse_log,
        "smape_pct": smape,
        "n_obs": int(len(maturities)),
    }
    return params, metrics


random_date = steepener_data.sample(n=1).index[0]
min_maturity = 20
max_maturity = 500

params, metrics = nss_metrics_for_date(steepener_data, random_date, min_maturity=min_maturity, max_maturity=max_maturity, verbosity=True)

print(f"\nNSS params ({random_date}):", params)
print(f"Metrics for {random_date}:", metrics)

plot_nss_fits(steepener_data, dates=[random_date], min_maturity=min_maturity, max_maturity=max_maturity, grid_step=10)


In [ ]:
def nss_price_spread_for_date(df: pd.DataFrame, date_str: str, min_maturity: int = 20):
    """Return per-contract spreads for a single date."""
    if date_str not in df.index:
        raise ValueError(f"{date_str} not found in DataFrame index.")
    day_data = df.loc[date_str]
    day_data = day_data[day_data["time_to_maturity"] >= min_maturity]
    if day_data.empty:
        return []
    maturities = day_data["time_to_maturity"].to_numpy()
    prices = day_data["price"].to_numpy()
    params = NSS_estimation(maturities, prices, min_maturity=min_maturity)
    modeled = np.exp(NSS_rate(maturities, params))
    entries = []
    for contract, real_price, model_price in zip(day_data["contract"].to_numpy(), prices, modeled):
        spread = model_price - real_price
        #rel error bps
        rel_error_bps = spread / real_price * 10000  if real_price != 0 else np.nan
        entries.append({
            "contract": contract,
            "real_price": float(real_price),
            "model_price": float(model_price),
            "spread": float(spread),
            "error_bps": float(rel_error_bps)
        })
    return entries

# Example: spreads for a single date in the steepener window
random_date = steepener_data.sample(n=1).index[0]
# random_date = '2025-05-28'
# Add plot for curve for random date and date after random_date
random_date_next = (pd.to_datetime(random_date) + pd.tseries.offsets.BDay(1)).strftime('%Y-%m-%d')
plot_term_structure_evolution(steepener_data, dates=[random_date, random_date_next], figsize=(8,5))
print(random_date)
# example_spreads = nss_price_spread_for_date(steepener_data, random_date, min_maturity=10)
# print(pd.DataFrame(example_spreads))

# Backtest start

In [11]:
backtest_data = pd.read_csv(f'data/{commo_ticker}_NSS_price_spreads.csv')


# Diagnostic Analysis: NSS Curve as Trading Signal
This section systematically diagnoses whether NSS curve changes (betas) are tradable signals.
Focus on empirical evidence, not strategy.

In [23]:
import scipy.stats as stats

# Prepare dataset from CSV
nss_dataset = backtest_data.copy()
nss_dataset['date'] = pd.to_datetime(nss_dataset['date'])
nss_dataset = nss_dataset.sort_values(['date', 'time_to_maturity']).reset_index(drop=True)

print("\n" + "="*70)
print(f"NSS Curve Dataset Loaded: {commo_ticker}")
print("="*70)
print(f"Records: {len(nss_dataset)}")
print(f"Date range: {nss_dataset['date'].min()} to {nss_dataset['date'].max()}")
print(f"Unique dates: {nss_dataset['date'].nunique()}")
print(f"Unique contracts: {nss_dataset['contract'].nunique()}")
print(f"\nColumns: {list(nss_dataset.columns)}")
print(f"\nData sample:")
print(nss_dataset.head(10))


NSS Curve Dataset Loaded: KC
Records: 51157
Date range: 1989-01-03 00:00:00 to 2025-11-13 00:00:00
Unique dates: 9170
Unique contracts: 190

Columns: ['date', 'contract', 'time_to_maturity', 'open_interest', 'real_price', 'model_price', 'spread', 'error_bps', 'beta0', 'beta1', 'beta2', 'beta3', 'theta1', 'theta2']

Data sample:
        date contract  time_to_maturity  open_interest  real_price  \
0 1989-01-03    H1989                51            NaN      165.90   
1 1989-01-03    K1989                96            NaN      158.80   
2 1989-01-03    N1989               141            NaN      154.81   
3 1989-01-03    U1989               186            NaN      152.00   
4 1989-01-03    Z1989               251            NaN      149.50   
5 1989-01-04    H1989                50            NaN      160.13   
6 1989-01-04    K1989                95            NaN      155.06   
7 1989-01-04    N1989               140            NaN      152.00   
8 1989-01-04    U1989               185

## 1. Signal Horizon Diagnosis
Analyze 1-day, 5-day, 10-day beta changes for mean-reversion and predictability.

# Diagnostic Analysis (Corrected): Returns-Based Beta Attribution

**Approach:**
1. Compute log returns at horizons 1d, 5d, 10d for each contract.
2. Define instruments:
   - **Outrights**: individual contract returns (baseline).
   - **Maturity spreads**: nearest contracts to target TTM (30d, 90d, 180d, 360d) and their differences.
   - **Butterflies**: level-neutral (β₀-minimized) and curvature (β₂-targeted).
3. Regress returns on beta changes, check for implausible R² (>0.3 flags an error).
4. Compare instruments across horizons with Sharpe, turnover, and beta exposures.

**Key sanity rules:**
- R² on daily returns should be <0.15–0.25 (if >0.3, something is wrong).
- SNR on returns should be <10 (if >20, data leak or miscalculation).
- Show exact instrument construction logic and why spreads succeed or fail.

In [30]:
# Step 1: Compute log returns for all contracts and horizons
print("\n" + "="*80)
print("STEP 1: LOG RETURNS COMPUTATION")
print("="*80)

def compute_log_returns(dataset, horizons=[1, 5, 10]):
    """
    Compute log returns r_t(h) = log(P_t / P_{t-h}) for each contract and horizon.
    
    Returns: dict of DataFrames, keyed by horizon.
    Each DataFrame: index=date, columns=contracts, values=log returns (%).
    """
    returns_by_horizon = {}
    
    for horizon in horizons:
        records = []
        
        for contract in dataset['contract'].unique():
            contract_data = dataset[dataset['contract'] == contract].sort_values('date').reset_index(drop=True)
            
            # Compute log returns
            prices = contract_data['real_price'].values
            log_ret = 100 * np.log(prices[horizon:] / prices[:-horizon])
            dates = contract_data['date'].values[horizon:]
            
            for date, ret in zip(dates, log_ret):
                records.append({
                    'date': pd.Timestamp(date),
                    'contract': contract,
                    'return_pct': ret,
                })
        
        ret_df = pd.DataFrame(records).pivot_table(
            index='date', columns='contract', values='return_pct', aggfunc='first'
        )
        returns_by_horizon[horizon] = ret_df
    
    return returns_by_horizon

returns_dict = compute_log_returns(nss_dataset, horizons=[1, 5, 10])

for h in [1, 5, 10]:
    df = returns_dict[h]
    print(f"\n{h}d returns: {len(df)} dates, {df.shape[1]} contracts, {df.isna().sum().sum()} NaNs")
    print(f"  Mean return: {df.mean().mean():.6f}%, Vol: {df.mean().std():.6f}%")
    print(f"  Sample:\n{df.head(3)}")


STEP 1: LOG RETURNS COMPUTATION

1d returns: 9169 dates, 190 contracts, 1691143 NaNs
  Mean return: -0.019017%, Vol: 0.147492%
  Sample:
contract       H1989  H1990  H1991  H1992  H1993  H1994  H1995  H1996  H1997  \
date                                                                           
1989-01-04 -3.539921    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
1989-01-05  1.758015    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
1989-01-06 -3.274483    NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN   

contract    H1998  ...  Z2017  Z2018  Z2019  Z2020  Z2021  Z2022  Z2023  \
date               ...                                                    
1989-01-04    NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
1989-01-05    NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   
1989-01-06    NaN  ...    NaN    NaN    NaN    NaN    NaN    NaN    NaN   

contract    Z2024  Z2025  Z2026  
date                             
1989-01-04    NaN

In [34]:
# Step 2: Compute beta changes for each horizon
print("\n" + "="*80)
print("STEP 2: BETA CHANGES (Δβ)")
print("="*80)

def compute_beta_changes(dataset, horizons=[1, 5, 10]):
    """
    For each date and horizon h, compute Δβ_i = β_i(t) - β_i(t-h).
    Returns: dict of DataFrames indexed by date, columns [delta_beta0, delta_beta1, delta_beta2, delta_beta3].
    """
    beta_daily = dataset.groupby('date')[['beta0', 'beta1', 'beta2', 'beta3']].first().sort_index()
    
    delta_beta = {}
    for horizon in horizons:
        diffs = {}
        for col in ['beta0', 'beta1', 'beta2', 'beta3']:
            diffs[f'delta_{col}'] = beta_daily[col].diff(horizon)
        delta_beta[horizon] = pd.DataFrame(diffs)
    
    return delta_beta

delta_beta_dict = compute_beta_changes(nss_dataset, horizons=[1, 5, 10])

for h in [1, 5, 10]:
    df = delta_beta_dict[h]
    print(f"\n{h}d beta changes: {len(df)} obs, {df.isna().sum().sum()} NaNs")
    print(df.describe().round(6))


STEP 2: BETA CHANGES (Δβ)

1d beta changes: 9170 obs, 4 NaNs
       delta_beta0  delta_beta1  delta_beta2  delta_beta3
count  9169.000000  9169.000000  9169.000000  9169.000000
mean      0.000128    -0.000037     0.000080    -0.000181
std       0.355906     0.369117     0.227868     0.996569
min      -5.355946    -6.033819    -4.643523   -16.573987
25%      -0.018720    -0.012947    -0.019952    -0.005789
50%      -0.000336     0.000405    -0.000537     0.000900
75%       0.018236     0.014483     0.017918     0.007242
max       5.788745     5.682124     3.708410    15.575073

5d beta changes: 9170 obs, 20 NaNs
       delta_beta0  delta_beta1  delta_beta2  delta_beta3
count  9165.000000  9165.000000  9165.000000  9165.000000
mean      0.000508    -0.000019     0.000335    -0.000489
std       0.460300     0.476396     0.303447     1.293280
min      -7.629933    -9.757002    -5.982912   -26.878071
25%      -0.048253    -0.036195    -0.050776    -0.031430
50%      -0.001610     0.000219 

In [32]:
# Step 3: Define instruments—outrights, spreads, and butterflies
print("\n" + "="*80)
print("STEP 3: INSTRUMENT DEFINITIONS")
print("="*80)

def build_instruments(dataset, horizons=[1, 5, 10]):
    """
    Build return series for:
    1. OUTRIGHTS: each contract's log return (as is).
    2. MATURITY SPREADS: select nearest contracts to targets (30d, 90d, 180d, 360d),
       compute spread returns as (P_long - P_short) change over horizon.
    3. BUTTERFLIES: level-neutral (β₀-min) and curvature (β₂-target) constructs.
    
    Returns: dict of dicts: instruments[horizon]['instrument_name'] = return_series
    """
    instruments = {}
    
    for horizon in horizons:
        h_instruments = {}
        ret_df = returns_dict[horizon]
        
        # --- OUTRIGHTS ---
        for col in ret_df.columns:
            h_instruments[f'outright_{col}'] = ret_df[col].copy()
        
        # --- MATURITY SPREADS ---
        # Targets in business days
        targets = {'30d': 30, '90d': 90, '180d': 180, '360d': 360}
        
        # For each date, select nearest contract to each target
        unique_dates = sorted(dataset['date'].unique())
        
        for short_lbl, long_lbl in [('30d', '90d'), ('30d', '180d'), ('90d', '180d'), ('180d', '360d')]:
            spread_returns = []
            short_target = targets[short_lbl]
            long_target = targets[long_lbl]
            
            for date in unique_dates:
                date_data = dataset[dataset['date'] == date]
                
                # Find nearest to short_target
                short_candidates = date_data[date_data['time_to_maturity'] >= 10]  # min maturity
                if len(short_candidates) == 0:
                    continue
                short_idx = (short_candidates['time_to_maturity'] - short_target).abs().idxmin()
                short_ttm = short_candidates.loc[short_idx, 'time_to_maturity']
                short_price = short_candidates.loc[short_idx, 'real_price']
                short_contract = short_candidates.loc[short_idx, 'contract']
                
                # Find nearest to long_target
                long_candidates = date_data[date_data['time_to_maturity'] >= 10]
                if len(long_candidates) == 0:
                    continue
                long_idx = (long_candidates['time_to_maturity'] - long_target).abs().idxmin()
                long_ttm = long_candidates.loc[long_idx, 'time_to_maturity']
                long_price = long_candidates.loc[long_idx, 'real_price']
                long_contract = long_candidates.loc[long_idx, 'contract']
                
                # Skip if targets too far or legs overlap
                if abs(short_ttm - short_target) > 30 or abs(long_ttm - long_target) > 30:
                    continue
                if short_ttm >= long_ttm:
                    continue
                
                spread_returns.append({
                    'date': date,
                    'short_contract': short_contract,
                    'long_contract': long_contract,
                    'short_ttm': short_ttm,
                    'long_ttm': long_ttm,
                    'spread_price': long_price - short_price,
                })
            
            if len(spread_returns) < 2:
                print(f"  ⚠ {short_lbl}_v_{long_lbl} @ {horizon}d: insufficient data ({len(spread_returns)} dates)")
                continue
            
            spread_df = pd.DataFrame(spread_returns).set_index('date').sort_index()
            
            # Compute changes in spread over horizon
            if horizon == 1:
                spread_ret = spread_df['spread_price'].diff(1)
            else:
                spread_ret = spread_df['spread_price'].diff(horizon)
            
            h_instruments[f'spread_{short_lbl}_v_{long_lbl}'] = spread_ret.copy()
            print(f"  ✓ spread_{short_lbl}_v_{long_lbl} @ {horizon}d: {spread_ret.notna().sum()} obs")
        
        instruments[horizon] = h_instruments
    
    return instruments

instruments = build_instruments(nss_dataset, horizons=[1, 5, 10])

print(f"\nTotal instruments built:")
for h in [1, 5, 10]:
    print(f"  {h}d: {len(instruments[h])} series")


STEP 3: INSTRUMENT DEFINITIONS
  ✓ spread_30d_v_90d @ 1d: 7889 obs
  ✓ spread_30d_v_180d @ 1d: 7941 obs
  ✓ spread_90d_v_180d @ 1d: 8650 obs
  ⚠ 180d_v_360d @ 1d: insufficient data (0 dates)
  ✓ spread_30d_v_90d @ 5d: 7885 obs
  ✓ spread_30d_v_180d @ 5d: 7937 obs
  ✓ spread_90d_v_180d @ 5d: 8646 obs
  ⚠ 180d_v_360d @ 5d: insufficient data (0 dates)
  ✓ spread_30d_v_90d @ 10d: 7880 obs
  ✓ spread_30d_v_180d @ 10d: 7932 obs
  ✓ spread_90d_v_180d @ 10d: 8641 obs
  ⚠ 180d_v_360d @ 10d: insufficient data (0 dates)

Total instruments built:
  1d: 193 series
  5d: 193 series
  10d: 193 series


In [36]:
# Step 4: Returns-based regression with sanity checks
print("\n" + "="*80)
print("STEP 4: RETURNS-BASED REGRESSION (r_t ~ ΔBeta)")
print("="*80)

import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def regress_returns_on_beta_changes(returns_series, delta_beta_dict, horizon):
    """
    Regress log returns on beta changes.
    
    Input:
      returns_series: pd.Series indexed by date
      delta_beta_dict: dict keyed by horizon, containing DataFrame with [delta_beta0/1/2/3] indexed by date
      horizon: int (1, 5, or 10)
    
    Output:
      {
        'r_squared': float,
        'coeffs': dict of beta0/1/2/3 coefficients,
        'tstats': dict of t-statistics,
        'pvalues': dict of p-values,
        'snr': float (coefficient / residual std),
        'warnings': list of strings
      }
    """
    # Align returns and beta changes
    delta_beta = delta_beta_dict[horizon].copy()
    returns = returns_series.copy()
    
    aligned = pd.concat([returns.rename('returns'), delta_beta], axis=1).dropna()
    
    if len(aligned) < 10:
        return {'error': f"Insufficient obs: {len(aligned)}"}
    
    X = aligned[['delta_beta0', 'delta_beta1', 'delta_beta2', 'delta_beta3']]
    y = aligned['returns']
    
    # Add constant
    X_with_const = sm.add_constant(X)
    model = sm.OLS(y, X_with_const).fit()
    
    result = {
        'n_obs': len(aligned),
        'r_squared': model.rsquared,
        'adj_r_squared': model.rsquared_adj,
        'coeffs': {
            'beta0': model.params['delta_beta0'],
            'beta1': model.params['delta_beta1'],
            'beta2': model.params['delta_beta2'],
            'beta3': model.params['delta_beta3'],
        },
        'tstats': {
            'beta0': model.tvalues['delta_beta0'],
            'beta1': model.tvalues['delta_beta1'],
            'beta2': model.tvalues['delta_beta2'],
            'beta3': model.tvalues['delta_beta3'],
        },
        'pvalues': {
            'beta0': model.pvalues['delta_beta0'],
            'beta1': model.pvalues['delta_beta1'],
            'beta2': model.pvalues['delta_beta2'],
            'beta3': model.pvalues['delta_beta3'],
        },
        'residual_std': model.resid.std(),
        'warnings': []
    }
    
    # SANITY CHECKS
    if result['r_squared'] > 0.3:
        result['warnings'].append(f"⚠️ R²={result['r_squared']:.3f} > 0.3 (suspiciously high for daily returns! check for data leakage)")
    
    # SNR = max absolute coefficient / residual std
    coeff_values = [abs(v) for v in result['coeffs'].values()]
    if len(coeff_values) > 0:
        max_coeff = max(coeff_values)
        snr = max_coeff / (result['residual_std'] + 1e-6)
        result['snr'] = snr
        if snr > 20:
            result['warnings'].append(f"⚠️ SNR={snr:.1f} > 20 (implausibly high! check for price-level contamination)")
    
    return result

# Run regressions for all instruments and horizons
regression_results = {}  # results[horizon][instrument_name] = result_dict

for horizon in [1, 5, 10]:
    regression_results[horizon] = {}
    
    for instrument_name, ret_series in instruments[horizon].items():
        result = regress_returns_on_beta_changes(ret_series, delta_beta_dict, horizon)
        
        if 'error' in result:
            print(f"  {instrument_name:30s} @ {horizon}d: {result['error']}")
            continue
        
        regression_results[horizon][instrument_name] = result
        
        # Print compact result
        r2_str = f"R²={result['r_squared']:.4f}"
        n_str = f"n={result['n_obs']}"
        sig_betas = [f"β{i}={result['coeffs'][f'beta{i}']:.4f}(t={result['tstats'][f'beta{i}']:.1f})" 
                     for i in range(4) if abs(result['tstats'][f'beta{i}']) > 1.96]
        
        if len(result['warnings']) > 0:
            warning_str = " " + " ".join(result['warnings'])
        else:
            warning_str = ""
        
        print(f"  {instrument_name:30s} @ {horizon}d: {r2_str:20s} {n_str:10s} {warning_str}")


STEP 4: RETURNS-BASED REGRESSION (r_t ~ ΔBeta)
  outright_H1989                 @ 1d: R²=0.9795            n=39        ⚠️ R²=0.979 > 0.3 (suspiciously high for daily returns! check for data leakage) ⚠️ SNR=253.5 > 20 (implausibly high! check for price-level contamination)
  outright_H1990                 @ 1d: R²=0.9259            n=279       ⚠️ R²=0.926 > 0.3 (suspiciously high for daily returns! check for data leakage) ⚠️ SNR=211.8 > 20 (implausibly high! check for price-level contamination)
  outright_H1991                 @ 1d: R²=0.9550            n=278       ⚠️ R²=0.955 > 0.3 (suspiciously high for daily returns! check for data leakage) ⚠️ SNR=303.6 > 20 (implausibly high! check for price-level contamination)
  outright_H1992                 @ 1d: R²=0.9198            n=278       ⚠️ R²=0.920 > 0.3 (suspiciously high for daily returns! check for data leakage) ⚠️ SNR=264.5 > 20 (implausibly high! check for price-level contamination)
  outright_H1993                 @ 1d: R²=0.8541

In [37]:
# Step 5: Comparison table across horizons and instruments
print("\n" + "="*80)
print("STEP 5: COMPARISON TABLE")
print("="*80)

# Flatten results into a single table
comparison_rows = []

for horizon in [1, 5, 10]:
    for instrument_name, result in regression_results[horizon].items():
        if 'error' in result:
            continue
        
        row = {
            'Horizon (d)': horizon,
            'Instrument': instrument_name,
            'n_obs': result['n_obs'],
            'R²': result['r_squared'],
            'Adj.R²': result['adj_r_squared'],
            'Residual_Std': result['residual_std'],
            'SNR': result.get('snr', None),
            'β₀_coeff': result['coeffs']['beta0'],
            'β₀_tstat': result['tstats']['beta0'],
            'β₀_pval': result['pvalues']['beta0'],
            'β₁_coeff': result['coeffs']['beta1'],
            'β₁_tstat': result['tstats']['beta1'],
            'β₁_pval': result['pvalues']['beta1'],
            'β₂_coeff': result['coeffs']['beta2'],
            'β₂_tstat': result['tstats']['beta2'],
            'β₂_pval': result['pvalues']['beta2'],
            'β₃_coeff': result['coeffs']['beta3'],
            'β₃_tstat': result['tstats']['beta3'],
            'β₃_pval': result['pvalues']['beta3'],
            'Warnings': "; ".join(result['warnings']) if result['warnings'] else "OK",
        }
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).sort_values(['Horizon (d)', 'R²'], ascending=[True, False])

# Display key columns
display_cols = ['Horizon (d)', 'Instrument', 'n_obs', 'R²', 'SNR', 
                'β₁_coeff', 'β₁_tstat', 'β₁_pval',
                'β₂_coeff', 'β₂_tstat', 'β₂_pval', 'Warnings']

print("\nComparison Table (sorted by horizon, then R²):")
print(comparison_df[display_cols].to_string(index=False))

# Summary statistics by horizon
print("\n" + "="*80)
print("HORIZON-LEVEL SUMMARY")
print("="*80)

for horizon in [1, 5, 10]:
    h_data = comparison_df[comparison_df['Horizon (d)'] == horizon]
    
    if len(h_data) == 0:
        print(f"\n{horizon}d: No valid results")
        continue
    
    mean_r2 = h_data['R²'].mean()
    max_r2 = h_data['R²'].max()
    n_sig_beta1 = (h_data['β₁_pval'] < 0.05).sum()
    n_sig_beta2 = (h_data['β₂_pval'] < 0.05).sum()
    n_warn = (h_data['Warnings'] != 'OK').sum()
    
    print(f"\n{horizon}d Horizon:")
    print(f"  Instruments: {len(h_data)} valid")
    print(f"  Mean R²: {mean_r2:.4f}, Max R²: {max_r2:.4f}")
    print(f"  Significant β₁ (p<0.05): {n_sig_beta1} instruments")
    print(f"  Significant β₂ (p<0.05): {n_sig_beta2} instruments")
    print(f"  With warnings: {n_warn} instruments")


STEP 5: COMPARISON TABLE

Comparison Table (sorted by horizon, then R²):
 Horizon (d)        Instrument  n_obs       R²         SNR   β₁_coeff   β₁_tstat       β₁_pval  β₂_coeff   β₂_tstat       β₂_pval                                                                                                                                                    Warnings
           1    outright_Z2026     15 0.999196 1416.940446  33.122384  19.854805  2.304482e-09 27.068837  46.337606  5.278560e-13 ⚠️ R²=0.999 > 0.3 (suspiciously high for daily returns! check for data leakage); ⚠️ SNR=1416.9 > 20 (implausibly high! check for price-level contamination)
           1    outright_K2022    279 0.998419 1133.177900  58.925810  45.321421 2.535195e-129 25.633192  39.649946 1.627635e-115 ⚠️ R²=0.998 > 0.3 (suspiciously high for daily returns! check for data leakage); ⚠️ SNR=1133.2 > 20 (implausibly high! check for price-level contamination)
           1    outright_U2026     77 0.998050  845.496348  38.19405

In [ ]:
# Step 6: Final verdict
print("\n" + "="*80)
print("FINAL VERDICT: ARE NSS BETAS TRADABLE?")
print("="*80)

# Count failures and successes by criterion
failures = {
    'r2_too_high': (comparison_df['R²'] > 0.3).sum(),
    'snr_too_high': (comparison_df['SNR'] > 20).sum(),
    'warnings_present': (comparison_df['Warnings'] != 'OK').sum(),
    'no_sig_betas': (comparison_df['β₁_pval'] >= 0.05) & (comparison_df['β₂_pval'] >= 0.05),
}

print("\n⚠️  SANITY CHECK FAILURES:")
print(f"  R² > 0.3 (suspiciously high): {failures['r2_too_high']} instruments")
print(f"  SNR > 20 (implausibly high): {failures['snr_too_high']} instruments")
print(f"  Total warnings: {failures['warnings_present']} instruments")

# Identify any instruments passing sanity checks
clean_results = comparison_df[
    (comparison_df['R²'] <= 0.3) & 
    ((comparison_df['SNR'] <= 20) | (comparison_df['SNR'].isna())) &
    (comparison_df['Warnings'] == 'OK')
]

if len(clean_results) == 0:
    print("\n❌ **VERDICT: NO TRADABLE SIGNALS FOUND**")
    print("\nAll instruments trigger sanity check warnings.")
    print("This suggests:")
    print("  1. NSS beta changes are NOT forecastable from historical price moves")
    print("  2. Beta coefficients are statistically significant but NOT practically predictive")
    print("  3. Any apparent predictability is likely statistical artifact, not economic edge")
    print("\n→ Do NOT trade NSS curve factors as standalone signals.")
else:
    print(f"\n✓ CLEAN RESULTS: {len(clean_results)} instrument(s) pass sanity checks\n")
    print(clean_results[['Horizon (d)', 'Instrument', 'R²', 'β₁_pval', 'β₂_pval']].to_string(index=False))
    print("\nThese may merit further investigation, but manage expectations:")
    print("  - R² still very low (< 0.3), so limited predictive power")
    print("  - Significance ≠ tradability; need Sharpe > 1, turnover < 20%")
    print("  - Likely still too weak for live trading; consider ensemble or multi-factor")
    """
    Compute 1d, 5d, 10d beta changes.
    Measure:
    - Autocorrelation (mean-reversion)
    - Variance (signal strength)
    - Predictability (correlation between changes at different lags)
    """
    print("\n" + "="*70)
    print("SIGNAL HORIZON DIAGNOSIS")
    print("="*70)
    
    # Get daily beta values (one per date)
    beta_daily = dataset.groupby('date')[['beta0', 'beta1', 'beta2', 'beta3']].first().reset_index()
    beta_daily = beta_daily.set_index('date').sort_index()
    
    betas = ['beta0', 'beta1', 'beta2', 'beta3']
    horizons = [1, 5, 10]
    
    results = []
    
    for beta_name in betas:
        series = beta_daily[beta_name]
        
        # Compute changes
        changes = {}
        for h in horizons:
            changes[h] = series.diff(h).dropna()
        
        print(f"\n{beta_name}:")
        print(f"  Daily mean: {series.mean():.6f}, std: {series.std():.6f}")
        
        for h in horizons:
            ch = changes[h]
            acf_1 = ch.autocorr(lag=1)  # Mean-reversion indicator
            acf_5 = ch.autocorr(lag=5)
            
            print(f"  {h}d change - mean: {ch.mean():.6f}, std: {ch.std():.6f}, " +
                  f"AC(1): {acf_1:.4f}, AC(5): {acf_5:.4f}")
            
            results.append({
                'beta': beta_name,
                'horizon': h,
                'mean_change': ch.mean(),
                'std_change': ch.std(),
                'autocorr_1': acf_1,
                'autocorr_5': acf_5,
                'skew': ch.skew(),
                'kurt': ch.kurtosis(),
            })
    
    results_df = pd.DataFrame(results)
    
    # Interpretation
    print("\n" + "-"*70)
    print("INTERPRETATION:")
    print("-"*70)
    for beta_name in betas:
        subset = results_df[results_df['beta'] == beta_name]
        print(f"\n{beta_name}:")
        
        # Check for mean-reversion (negative AC)
        ac1_mean = subset['autocorr_1'].mean()
        if ac1_mean < -0.1:
            print(f"  ✓ MEAN-REVERTING (AC(1)={ac1_mean:.3f})")
        elif ac1_mean > 0.3:
            print(f"  ✗ TRENDING (AC(1)={ac1_mean:.3f})")
        else:
            print(f"  ~ RANDOM-WALK-ISH (AC(1)={ac1_mean:.3f})")
        
        # Check volatility
        vol_1d = subset[subset['horizon'] == 1]['std_change'].values[0]
        vol_10d = subset[subset['horizon'] == 10]['std_change'].values[0]
        print(f"  Volatility: 1d={vol_1d:.6f}, 10d={vol_10d:.6f}")
    
    return results_df

horizon_results = analyze_beta_horizons(nss_dataset)


FINAL VERDICT: ARE NSS BETAS TRADABLE?

⚠️  SANITY CHECK FAILURES:
  R² > 0.3 (suspiciously high): 575 instruments
  SNR > 20 (implausibly high): 574 instruments
  Total warnings: 576 instruments

✓ CLEAN RESULTS: 2 instrument(s) pass sanity checks

 Horizon (d)        Instrument       R²  β₁_pval       β₂_pval
           5 spread_90d_v_180d 0.267214      0.0 1.840234e-106
          10 spread_90d_v_180d 0.286030      0.0 1.150227e-149

These may merit further investigation, but manage expectations:
  - R² still very low (< 0.3), so limited predictive power
  - Significance ≠ tradability; need Sharpe > 1, turnover < 20%
  - Likely still too weak for live trading; consider ensemble or multi-factor
